<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
The decision this improves is "which pages should an editor review first this week?" a which-ones-first question, not a yes/no question about one page in isolation. The output is a priority score per page; the editor works down the ranked list starting from the top. Classification (declining vs. not) is one ingredient of the score, but the deliverable itself is an ordering, which makes this a scoring/ranking task.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

No direct "worth reviewing" label exists in this data — nobody logged real editor decisions. So I'm combining two observed signals into a proxy: is_declining_label (from trend_direction, measured over real 30-day windows) AND measurable_opportunity (real traffic: impressions ≥ 100 and sessions > 0). I call the combination worth_reviewing. trend_direction/trend_pct will never be used as model features later — they only exist here to define the target.

In [1]:
import pandas as pd

df = pd.read_csv('/content_refresh_anonymized.csv')

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)
df['worth_reviewing'] = ((df['is_declining_label'] == 1) & (df['measurable_opportunity'] == 1)).astype(int)

print(f"{df['is_declining_label'].mean()*100:.1f}% of all pages are declining")
print(f"{df['worth_reviewing'].mean()*100:.1f}% of all pages are 'worth reviewing' (declining + real traffic)")
df[['content_id','client_id','trend_direction','impressions_90d','is_declining_label','worth_reviewing']].head()

54.2% of all pages are declining
43.8% of all pages are 'worth reviewing' (declining + real traffic)


,content_id,client_id,trend_direction,impressions_90d,is_declining_label,worth_reviewing
0,content_304f48230142,client_f369cb89fc,down,3803,1,1
1,content_a1fb4e703a9e,client_4e07408562,down,15320,1,1
2,content_9aa793d4d895,client_7f2253d7e2,down,12581,1,1
3,content_331d6c4de07b,client_19581e27de,stable,11751,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140,1,1


## 3. Success metric
precision@50. Of the top 50 pages ranked by opportunity score, what percent are truly worth_reviewing? This mirrors the real action — an editor works down a ranked list, not the whole dataset. I computed it against a naive baseline (rank by traffic alone) below, to know what "good" needs to beat.

In [2]:
baseline_top50 = df.sort_values('impressions_90d', ascending=False).head(50)
precision_at_50 = baseline_top50['worth_reviewing'].mean() * 100
overall_rate = df['worth_reviewing'].mean() * 100

print(f"Baseline (rank by traffic alone): {precision_at_50:.1f}% of top 50 are truly worth reviewing")
print(f"Overall base rate across all 30,000 pages: {overall_rate:.1f}%")


Baseline (rank by traffic alone): 42.0% of top 50 are truly worth reviewing
Overall base rate across all 30,000 pages: 43.8%


## 4. The unit of analysis, as a real dataframe

One row = one content page (content_id), belonging to one client (client_id). 30,000 pages across 32 clients, each with trailing-90-day metrics.

In [3]:
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Unique pages: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")

df[['content_id', 'client_id', 'content_type', 'impressions_90d',
    'avg_position', 'engagement_rate', 'trend_direction', 'worth_reviewing']].head(10)

Shape: 30,000 rows, 47 columns
Unique pages: 30,000
Unique clients: 32


,content_id,client_id,content_type,impressions_90d,avg_position,engagement_rate,trend_direction,worth_reviewing
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,8.5,0.00,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,7.0,0.00,down,0
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,21.2,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,46.0,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,4.9,0.00,down,1


## 5. Why ML beats a fixed rule here

A single if-statement (e.g. "flag any page where trend_pct drops past -20%") treats a declining page with 20 impressions the same as one with 20,000 — it can't weigh decline against impact. A naive rule based on traffic alone also fails: ranking by traffic size showed almost no lift over random (42.0% vs. 43.8% base rate), proving traffic size and "worth reviewing" aren't the same thing. The signals interact and the "worth it" line isn't fixed — it shifts by content type and client. That's exactly the kind of tangled, multi-signal pattern ML is for; a fixed rule can't hold all of it at once.

In [4]:
declining = df[df['is_declining_label'] == 1].sort_values('impressions_90d', ascending=False)
print("High-traffic decliners:")
print(declining[['content_id','trend_pct','impressions_90d','worth_reviewing']].head(3))
print("\nLow-traffic decliners:")
print(declining[['content_id','trend_pct','impressions_90d','worth_reviewing']].tail(3))


High-traffic decliners:
                 content_id  trend_pct  impressions_90d  worth_reviewing
6653   content_5fe46e04994d      -44.8           517715                1
26844  content_8c19996aa890      -44.5           509252                1
21819  content_4c36c775b818      -33.2           463103                1

Low-traffic decliners:
                 content_id  trend_pct  impressions_90d  worth_reviewing
5205   content_c92f573e549b     -100.0                1                0
14673  content_789dcb2aef9e     -100.0                1                0
18655  content_e9bc92689a75     -100.0                1                0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.